# Limits of detection and quantitation (LOD and LOQ)

**Question:** what is the lowest copper concentration the method can reliably
*detect*, and the lowest it can *quantify*, and which of the common ways of
estimating them should the lab report?

- The **LOD** is the lowest concentration that can be told apart from a blank with
  reasonable confidence. Below it, the lab reports "not detected".
- The **LOQ** is the lowest concentration that can be measured with acceptable
  precision. Between LOD and LOQ, Cu is present but the number is not reliable.

There are several recipes, and on the same data they can disagree by a factor of
ten. This notebook runs the two most common ones and explains the difference.

> **Data and assumptions.** All data are synthetic, made by `scripts/make_data.py`:
> copper (Cu) in water by ICP-OES. `blanks.csv` holds 30 procedural blanks (10 on each
> of 3 days) taken through the whole method, in µg/L. `calibration.csv` holds 6
> standards from 0 to 200 µg/L in triplicate, as intensity in counts per second (cps).
> The blanks have a small Cu contamination from reagents and labware; the standards
> are clean solutions. No real laboratory data are used.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from labqc.validation import intermediate_precision, linearity, lod_loq

DATA = Path("..") / "data" / "synthetic"
if not DATA.exists():  # run from the repository root
    DATA = Path("data") / "synthetic"

blanks = pd.read_csv(DATA / "blanks.csv")
cal = pd.read_csv(DATA / "calibration.csv")

# One quiet chart style for the handbook: one series colour, grey references.
BLUE, GREY, INK = "#2a78d6", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})

print(f"Blanks: {len(blanks)} results over {blanks['day'].nunique()} days")
print(f"Calibration: {cal['cu_ug_L'].nunique()} levels, "
      f"{cal['cu_ug_L'].min():.0f} to {cal['cu_ug_L'].max():.0f} µg/L, "
      f"{len(cal)} solutions")

## Route 1: from blank results

The Eurachem method validation guide estimates both limits from the standard
deviation s<sub>0</sub> of blank results (or of samples with a very low
concentration) measured through the whole procedure:

$$
\text{LOD} = 3\,s_0, \qquad \text{LOQ} = 10\,s_0
$$

The factor 3 keeps the risk of a false "detected" and of a false "not detected" low,
about 7 % each if the errors are normal and s<sub>0</sub> is well known (3.3 s<sub>0</sub>,
which ICH uses, gives 5 % each). The factor 10 means a result at the LOQ has a
relative standard deviation of about 10 %.

The blanks must be **independent**: separate preparations, ideally on different
days, so that s<sub>0</sub> includes everything that varies in routine work.

In [ ]:
s0 = blanks["cu_ug_L"].std(ddof=1)
print(f"Blank mean {blanks['cu_ug_L'].mean():.2f} µg/L, "
      f"s0 = {s0:.3f} µg/L (n = {len(blanks)})")
print(f"LOD = 3 s0  = {3 * s0:.2f} µg/L")
print(f"LOQ = 10 s0 = {10 * s0:.2f} µg/L")

### Why several days matter

The same calculation on each day's ten blanks alone:

In [ ]:
by_day = (
    blanks.groupby("day")["cu_ug_L"]
    .agg(n="size", mean="mean", s0=lambda v: v.std(ddof=1))
    .assign(LOD=lambda d: 3 * d["s0"], LOQ=lambda d: 10 * d["s0"])
)
display(by_day.round(2))

anova = intermediate_precision(blanks, value_col="cu_ug_L", group_col="day")
print(f"Between-day F test: p = {anova.p_value:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
rng = np.random.default_rng(1)  # horizontal jitter only, so points do not overlap
for day, group in blanks.groupby("day"):
    x = day + rng.uniform(-0.12, 0.12, size=len(group))
    ax.scatter(x, group["cu_ug_L"], s=22, color=BLUE, zorder=3)
    ax.hlines(group["cu_ug_L"].mean(), day - 0.25, day + 0.25, color=INK, lw=1.5)
ax.axhline(0, color=GREY, lw=0.8)
ax.set_xticks(sorted(blanks["day"].unique()))
ax.set_xticklabels([f"Day {d}\nLOD {3 * s:.2f}" for d, s in by_day["s0"].items()])
ax.set_ylabel("Blank result, Cu (µg/L)")
ax.set_title("30 procedural blanks: ten per day, line = day mean")
plt.tight_layout()
plt.show()

One day's blanks would have given an LOD anywhere from about 0.6 to 1.0 µg/L. That
is the normal sampling scatter of a standard deviation from ten results, and the
reason to pool blanks from several days.

The ANOVA finds no significant difference between the days. That does not prove
there is none: three days are too few to detect a modest day-to-day effect. A
longer series of blanks, collected during routine work, gives a better s<sub>0</sub>.

Two more points from this data:

- **The blanks do not average zero.** Their mean, about 0.4 µg/L, is Cu from
  reagents and labware. The LOD describes the *scatter* of the blank; the blank
  *level* is a bias the lab either subtracts (a procedural blank correction) or
  includes in its uncertainty.
- **Negative results are kept.** Replacing them with zero or "< LOD" would shrink
  s<sub>0</sub> and make the LOD look better than it is.

## Route 2: from the calibration line

ICH Q2 uses the residual standard deviation of the calibration line, s<sub>y/x</sub>,
and its slope *b*:

$$
\text{LOD} = \frac{3.3\,s_{y/x}}{b}, \qquad \text{LOQ} = \frac{10\,s_{y/x}}{b}
$$

This is what `labqc.validation.lod_loq` computes. It needs no extra measurements,
which makes it popular. Fitted over the whole working range:

In [ ]:
line_all = linearity(cal["cu_ug_L"], cal["intensity_cps"])
lod_all, loq_all = lod_loq(line_all)
print(f"slope b = {line_all.slope:.0f} cps per µg/L, "
      f"s_y/x = {line_all.residual_sd:.0f} cps, r² = {line_all.r_squared:.4f}")
print(f"LOD = {lod_all:.2f} µg/L, LOQ = {loq_all:.2f} µg/L")

That LOD is five times the blank estimate. The residual plot shows why:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(cal["cu_ug_L"], line_all.residuals, s=22, color=BLUE, zorder=3)
ax.axhline(0, color=INK, lw=0.8)
ax.set_xlabel("Standard, Cu (µg/L)")
ax.set_ylabel("Residual (cps)")
ax.set_title("Calibration residuals widen with concentration")
plt.tight_layout()
plt.show()

The residuals fan out: the scatter at 100 and 200 µg/L is more than a hundred times
larger than at zero. One s<sub>y/x</sub> for the whole line is dominated by the top
standards, so it says almost nothing about the noise near zero, where the LOD lives.
The [linearity notebook](02_linearity.ipynb) returns to this (weighted regression).

The usual fix is to estimate the limits from a calibration **close to the
expected LOD** only:

In [ ]:
low = cal[cal["cu_ug_L"] <= 25]
line_low = linearity(low["cu_ug_L"], low["intensity_cps"])
lod_low, loq_low = lod_loq(line_low)
print(f"Standards 0 to 25 µg/L: s_y/x = {line_low.residual_sd:.0f} cps")
print(f"LOD = {lod_low:.2f} µg/L, LOQ = {loq_low:.2f} µg/L")

Now the estimate is *below* the blank route. Calibration standards are clean
solutions measured in one sequence: they do not see the digestion, the labware or
the day-to-day changes that real samples, and the procedural blanks, go through.

> **A units trap in `lod_loq`.** Its optional `blank_sd` argument is divided by the
> slope, so it must be in **signal units** (cps), not µg/L. Blank results already in
> µg/L give LOD = 3.3 × s<sub>0</sub> directly, with no slope.

In [ ]:
# The same blank SD through lod_loq: convert it to cps first.
lod_blank_ich, loq_blank_ich = lod_loq(line_all, blank_sd=s0 * line_all.slope)
print(f"blank_sd in cps:  LOD = {lod_blank_ich:.2f} µg/L (3.3 s0; ICH factor)")
wrong, _ = lod_loq(line_all, blank_sd=s0)
print(f"blank_sd in µg/L: LOD = {wrong:.5f} µg/L  "
      f"<- wrong by a factor of {line_all.slope:.0f}")

## Which estimate to report?

In [ ]:
summary = pd.DataFrame(
    [
        ("Blanks, 30 over 3 days (3 s0)", 3 * s0, 10 * s0),
        ("Blanks, one day only (3 s0)",
         f"{by_day['LOD'].min():.2f} to {by_day['LOD'].max():.2f}",
         f"{by_day['LOQ'].min():.2f} to {by_day['LOQ'].max():.2f}"),
        ("Calibration line, 0 to 200 µg/L (ICH)", lod_all, loq_all),
        ("Calibration line, 0 to 25 µg/L (ICH)", lod_low, loq_low),
    ],
    columns=["route", "LOD (µg/L)", "LOQ (µg/L)"],
).set_index("route")
display(summary.map(lambda v: f"{v:.2f}" if isinstance(v, float) else v))

For this method the blank route, **LOD ≈ 0.8 µg/L and LOQ ≈ 2.6 µg/L**, is the one
to report:

- it is the only estimate built from material that went through the whole
  procedure, as samples do;
- the full-range calibration overstates the limits about fivefold, because of
  the widening residuals;
- the low-range calibration understates them, because clean standards miss the
  sample-preparation noise.

An LOQ is a claim, so check it: analyse a sample spiked near 2.6 µg/L several
times on different days and confirm the precision (and recovery) is acceptable
for the intended use. If it is not, raise the LOQ to a level where it is.

> **From the lab.** *[To write: how LOD and LOQ were set and checked for a method
> you validated, and what the assessor or the customer asked about them.]*

## References

- Eurachem, *The Fitness for Purpose of Analytical Methods* (method validation
  guide), section on LOD and LOQ.
- ICH Q2(R2), *Validation of Analytical Procedures*.
- ISO 11843, *Capability of detection*: the formal treatment of critical values
  and minimum detectable values.
- `labqc.validation`: [API documentation](https://mantas123456.github.io/lab-quality-toolkit/api/validation/).